# Evaluation and Figures: Tables and Figures of the Paper

This notebook reproduces the tables and figures of the paper *Comprehensive Evaluation of Monocular and Multi-view 3D Human Pose Estimation Models for Fitness Exercise Analysis*. It does not run any model: it reads the outputs of the previous notebooks, computes the evaluation results and verifies them against the values printed in the paper.

| Paper item | Section | Input read by the notebook |
|---|---|---|
| Table 4, Figs. 4 and 5 (influence of the 2D detector) | 3 | `results_per_video_<2D source>_overlap_hann_w243_s60.csv` of notebooks 01 and 02 (three 2D sources each) |
| Table 5, Figs. 6 and 8, Fig. 7 (monocular vs. multi-view) | 4 | 3D predictions of notebooks 01, 02 and 03 (RTMPose runs for the monocular models), Fit3D ground truth |
| Tables 6 and 7, Figs. 9, 10 and 11 (temporal filtering) | 5 | `smoothing_COMMON15_per_video.csv`, original and filtered predictions of notebook 04 |
| Fig. 2 (skeleton conventions) | 6 | none |

Figs. 1 and 3 and Tables 1 to 3 do not depend on any result and are not generated here.

## Evaluation protocol

- **Dataset:** the eight training subjects of Fit3D (`s03`, `s04`, `s05`, `s07`, `s08`, `s09`, `s10`, `s11`) and their 28 exercises; warm-up sequences are excluded. This gives **224 sequences per model**.
- **PA-MPJPE** (spatial accuracy): every frame of the prediction is aligned to the ground truth with a similarity transformation (rotation, translation and uniform scale), and the error is averaged over joints and frames. The analysis of the 2D detector uses the original 17-joint skeleton of TCPFormer and PoseMamba-L; every other comparison uses the common 15-joint skeleton (**COMMON15**), in millimetres.
- **MPJVE** (temporal consistency): mean error of the frame-to-frame joint displacement, in mm/frame, computed on pelvis-centred poses without any Procrustes alignment or scaling. It is produced by notebook 04.
- **Mean body velocity** (Figs. 9 and 10): visualization only; one scale factor per sequence, estimated from the original prediction, matches the size of the predicted body to the ground truth.

Every table and figure is checked against the paper. A value is reported as `OK` when it is within the tolerance of the check and as `CHECK` otherwise; the final section summarizes all checks.

## 1. Setup and Configuration

### Inputs

All inputs are looked up under a single `STORAGE_ROOT` (Google Drive in Colab, the folder `data` otherwise), with the same layout as in notebooks 01 to 04:

```text
STORAGE_ROOT/
├── Fit3D_2D/<subject>/gt/<exercise>.json            # Fit3D ground truth (joints3d_25)     [or VIDEOS/<subject>/<exercise>/joints3d_25.json]
├── camera_params/<subject>/...                      # calibration (rotation of camera 65906101)
├── Fit3D_TCPFormer/                                 # notebook 01
│   ├── results_per_video_<2D source>_overlap_hann_w243_s60.csv
│   └── pred3d_pose2d_rtmpose_overlap_hann_w243_s60/<subject>/<exercise>.npy
├── Fit3D_PoseMamba/                                 # notebook 02 (same structure)
├── Fit3D_MVGFormer/predictions_3d/<subject>/<exercise>.npy                              # notebook 03
└── Fit3D_temporal_filtering/                        # notebook 04
    ├── filtered_predictions/<model>/<filter and parameters>/<subject>/<exercise>.npy
    └── metrics/smoothing_COMMON15_per_video.csv
```

### Outputs

Everything generated here is written to `Fit3D_evaluation/` inside `STORAGE_ROOT`:

```text
Fit3D_evaluation/
├── tables/    # Tables 4 to 7 as LaTeX (.tex) and CSV
├── figures/   # figures as PNG and PDF
└── csv/       # COMMON15 PA-MPJPE per sequence and per joint, velocity curves
```

In [ ]:
# Detect the runtime and set the storage location

import os
import sys
from pathlib import Path

try:
    import google.colab   # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Directory of the notebook (the working directory when the notebook starts).
PROJECT_DIR = Path(globals().get("PROJECT_DIR") or os.getcwd())

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    # Change only this path if the project data are stored elsewhere.
    STORAGE_ROOT = Path("/content/drive/MyDrive")
else:
    # For local execution, place the data under ./data or change this path.
    STORAGE_ROOT = PROJECT_DIR / "data"

print("Environment:", "Google Colab" if IN_COLAB else "Local")
print("Storage root:", STORAGE_ROOT)

In [ ]:
# Imports

import json
import textwrap
from functools import lru_cache

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

# Editable text in the PDF files
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.family": "DejaVu Sans"})

In [ ]:
# Global configuration

SUBJECTS = ["s03", "s04", "s05", "s07", "s08", "s09", "s10", "s11"]

# 8 subjects x 28 non-warm-up exercises
EXPECTED_EXERCISES = 28
EXPECTED_SEQUENCES_PER_MODEL = len(SUBJECTS) * EXPECTED_EXERCISES

MODEL_ORDER = ["TCPFormer", "PoseMamba-L", "MVGFormer"]

# Camera used for all monocular experiments (frontal view).
MONO_CAMERA_ID = "65906101"

# Inference configuration of the monocular models (overlapping windows fused with a Hann weighting).
RUN_TAG = "overlap_hann_w243_s60"

# Dataset and calibration
GT_ROOT = STORAGE_ROOT / "Fit3D_2D"
VIDEOS_ROOT = STORAGE_ROOT / "VIDEOS"
CAMERA_ROOT = STORAGE_ROOT / "camera_params"

# Results of the previous notebooks. For each model a list of candidate folders is given;
# the first one that exists is used.
MODEL_ROOTS = {
    "TCPFormer": [
        STORAGE_ROOT / "Fit3D_TCPFormer",
        STORAGE_ROOT / "Fit3D_TCPFormer_TFG",
    ],
    "PoseMamba-L": [
        STORAGE_ROOT / "Fit3D_PoseMamba",
        STORAGE_ROOT / "Fit3D_PoseMamba_TFG",
        STORAGE_ROOT / "Fit3D_PoseMamba_L_TFG",
    ],
    "MVGFormer": [
        STORAGE_ROOT / "Fit3D_MVGFormer",
        STORAGE_ROOT / "Fit3D_MVGFormer_TFG",
        STORAGE_ROOT / "MVGFormer_results_TFG",
    ],
}

TEMPORAL_ROOT = STORAGE_ROOT / "Fit3D_temporal_filtering"
FILTERED_ROOT = TEMPORAL_ROOT / "filtered_predictions"
SMOOTHING_METRICS_CSV = TEMPORAL_ROOT / "metrics" / "smoothing_COMMON15_per_video.csv"

# Outputs
OUTPUT_ROOT = STORAGE_ROOT / "Fit3D_evaluation"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
CSV_DIR = OUTPUT_ROOT / "csv"

for folder in [TABLES_DIR, FIGURES_DIR, CSV_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

SAVE_PNG = True
SAVE_PDF = True
PNG_DPI = 300                  # the paper figures were exported at 600 dpi


def first_existing(paths):
    for path in paths:
        if Path(path).is_dir():
            return Path(path)
    return None


# Per-model settings
#   unit_scale : factor that converts the stored array to millimetres
#   gt_swap    : ground-truth side swap needed to match the model's joint order
#   gt_frame   : reference frame in which the velocity of the prediction is expressed
MODEL_CONFIGS = {
    "TCPFormer": {"num_joints": 17, "unit_scale": 1000.0, "gt_swap": "legs", "gt_frame": "camera"},
    "PoseMamba-L": {"num_joints": 17, "unit_scale": 1000.0, "gt_swap": "legs", "gt_frame": "camera"},
    "MVGFormer": {"num_joints": 15, "unit_scale": 1.0, "gt_swap": "arms", "gt_frame": "world"},
}

for model_name, cfg in MODEL_CONFIGS.items():
    cfg["result_root"] = first_existing(MODEL_ROOTS[model_name])

    if model_name == "MVGFormer":
        candidates = [root / "predictions_3d" for root in MODEL_ROOTS[model_name]]
    else:
        candidates = [root / f"pred3d_pose2d_rtmpose_{RUN_TAG}" for root in MODEL_ROOTS[model_name]]

    cfg["prediction_root"] = first_existing(candidates)

print(f"Subjects: {len(SUBJECTS)} | exercises: {EXPECTED_EXERCISES} | sequences per model: {EXPECTED_SEQUENCES_PER_MODEL}")
for model_name in MODEL_ORDER:
    print(f"{model_name:12s} results: {MODEL_CONFIGS[model_name]['result_root']}")
    print(f"{'':12s} predictions: {MODEL_CONFIGS[model_name]['prediction_root']}")
print("Ground truth :", GT_ROOT, "| exists:", GT_ROOT.is_dir() or VIDEOS_ROOT.is_dir())
print("Calibration  :", CAMERA_ROOT, "| exists:", CAMERA_ROOT.is_dir())
print("Smoothing metrics:", SMOOTHING_METRICS_CSV, "| exists:", SMOOTHING_METRICS_CSV.is_file())
print("Outputs      :", OUTPUT_ROOT)

## 2. Ground Truth, Conventions and Metrics

This section defines the functions shared by the following sections. They implement the evaluation protocol of the paper and are identical to those of notebook 04.

### Skeletons and ground-truth conventions

- **Monocular models (TCPFormer, PoseMamba-L):** 17-joint Human3.6M layout (`Pelvis, RHip, RKnee, RAnkle, LHip, LKnee, LAnkle, Spine, Neck, Nose, Head, LShoulder, LElbow, LWrist, RShoulder, RElbow, RWrist`). Joints 9 and 10 (`Nose` and `Head`) both derive from the COCO nose keypoint. The predictions are stored in metres and expressed in the frame of camera `65906101`.
- **MVGFormer:** 15-joint skeleton stored directly in COMMON15 order, in millimetres and in the world frame.
- **COMMON15:** the 15 joints that exist in both conventions (`Pelvis, RHip, RKnee, RAnkle, LHip, LKnee, LAnkle, Neck, Nose, LShoulder, LElbow, LWrist, RShoulder, RElbow, RWrist`). The monocular `Spine` and `Head` joints are discarded; the `Nose` joint is kept and matched with the nose of MVGFormer.
- **Ground truth:** the first 17 of the 25 Fit3D joints (metres), converted to COMMON15 and millimetres, with the left/right exchange required by each model (legs for the monocular models, arms for MVGFormer), and centred on the pelvis.

In [ ]:
# Ground-truth loading, COMMON15 conversion and metrics

COMMON15_INDICES = [0, 1, 2, 3, 4, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16]

COMMON15_NAMES = [
    "Pelvis", "RHip", "RKnee", "RAnkle", "LHip", "LKnee", "LAnkle",
    "Neck", "Nose", "LShoulder", "LElbow", "LWrist",
    "RShoulder", "RElbow", "RWrist",
]

# Bones of the COMMON15 skeleton (used for the body-size scale and for drawing).
SKELETON_15 = [
    (0, 1), (1, 2), (2, 3),
    (0, 4), (4, 5), (5, 6),
    (0, 7), (7, 8),
    (7, 9), (9, 10), (10, 11),
    (7, 12), (12, 13), (13, 14),
]

LEG_SWAP = ([1, 2, 3, 4, 5, 6], [4, 5, 6, 1, 2, 3])
ARM_SWAP = ([9, 10, 11, 12, 13, 14], [12, 13, 14, 9, 10, 11])


def is_warmup(name):
    return str(name).lower().startswith("warmup")


def root_align(x):
    x = np.asarray(x)
    return x - x[:, 0:1, :]


def swap_sides(pose, kind):
    '''Exchange the left and right limbs of a COMMON15 sequence ("legs" or "arms"; None keeps it unchanged).'''
    pose = np.array(pose, copy=True)
    if kind is None:
        return pose
    src, dst = LEG_SWAP if kind == "legs" else ARM_SWAP
    pose[:, src, :] = pose[:, dst, :].copy()
    return pose


def find_gt_json(subject, video):
    candidates = [
        GT_ROOT / subject / "gt" / f"{video}.json",
        GT_ROOT / subject / "gt (1)" / f"{video}.json",
        VIDEOS_ROOT / subject / video / "joints3d_25.json",
    ]
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError(
        f"Ground truth not found for {subject}/{video} under {GT_ROOT} or {VIDEOS_ROOT}"
    )


def find_camera_json(subject, video, camera_id=MONO_CAMERA_ID):
    base = CAMERA_ROOT / subject
    candidates = [
        base / camera_id / f"{video}.json",
        base / camera_id / f"{video} (1).json",
        base / "videos" / camera_id / f"{video}.json",
        base / "videos" / camera_id / f"{video} (1).json",
    ]
    for path in candidates:
        if path.is_file():
            return path
    if base.is_dir():
        for path in sorted(base.rglob("*.json")):
            if camera_id.lower() in str(path).lower() and video.lower() in path.stem.lower():
                return path
    raise FileNotFoundError(f"Calibration not found for {subject}/{camera_id}/{video} under {CAMERA_ROOT}")


@lru_cache(maxsize=None)
def camera_rotation(subject, video):
    '''Rotation R of the monocular camera, with X_cam = R @ X_world + T.'''
    path = find_camera_json(subject, video)
    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)

    rotation = np.asarray(data["extrinsics"]["R"], dtype=np.float64).reshape(3, 3)

    if not np.isfinite(rotation).all() or not np.allclose(rotation @ rotation.T, np.eye(3), atol=0.02):
        raise ValueError(f"Invalid rotation matrix in {path}")

    return rotation


def load_gt_common15(subject, video, gt_swap):
    '''Fit3D ground truth as COMMON15, in mm and pelvis-centred (world frame).

    gt_swap: "legs" (monocular models), "arms" (MVGFormer) or None (unmodified left/right labels).
    '''
    with open(find_gt_json(subject, video), "r", encoding="utf-8") as file:
        data = json.load(file)

    gt = np.asarray(data["joints3d_25"], dtype=np.float32)
    if gt.ndim == 4 and gt.shape[1] == 1:
        gt = gt[:, 0]
    if gt.ndim != 3 or gt.shape[1:] != (25, 3):
        raise ValueError(f"Unexpected ground-truth shape for {subject}/{video}: {gt.shape}")

    gt = gt[:, COMMON15_INDICES, :] * np.float32(1000.0)
    gt = swap_sides(gt, gt_swap)

    return root_align(gt).astype(np.float32)


def to_common15_mm(pose, cfg):
    '''Convert a stored prediction (or a filtered version of it) to COMMON15, mm, pelvis-centred.'''
    pose = np.asarray(pose, dtype=np.float32)
    if cfg["num_joints"] == 17:
        pose = pose[:, COMMON15_INDICES, :]
    return root_align(pose * np.float32(cfg["unit_scale"])).astype(np.float32)


def pa_errors(pred, gt, chunk=512):
    '''Frame-wise PA-MPJPE: Euclidean error (mm) of every joint after aligning each frame with a similarity
    transformation (batched SVD). Returns an array of shape (T, J).'''
    if pred.shape != gt.shape:
        raise ValueError(f"Shape mismatch: {pred.shape} vs {gt.shape}")

    num_frames, num_joints, _ = pred.shape
    errors = np.empty((num_frames, num_joints), dtype=np.float64)

    for start in range(0, num_frames, chunk):
        stop = min(start + chunk, num_frames)
        X = pred[start:stop].astype(np.float64)
        Y = gt[start:stop].astype(np.float64)

        mx = X.mean(axis=1, keepdims=True)
        my = Y.mean(axis=1, keepdims=True)
        X0, Y0 = X - mx, Y - my
        nx = np.linalg.norm(X0, axis=(1, 2))
        ny = np.linalg.norm(Y0, axis=(1, 2))

        valid = (nx > 1e-8) & (ny > 1e-8)
        aligned = X.copy()

        if valid.any():
            A = X0[valid] / nx[valid, None, None]
            B = Y0[valid] / ny[valid, None, None]
            U, s, Vt = np.linalg.svd(A.transpose(0, 2, 1) @ B)
            R = Vt.transpose(0, 2, 1) @ U.transpose(0, 2, 1)

            # Reflection correction. As in the evaluation of notebooks 01 to 03 (and of the paper), the scale uses
            # the sum of the singular values without changing the sign of the smallest one.
            flip = np.linalg.det(R) < 0
            if flip.any():
                Vt[flip, -1, :] *= -1
                R[flip] = Vt[flip].transpose(0, 2, 1) @ U[flip].transpose(0, 2, 1)

            scale = ny[valid] / nx[valid] * s.sum(axis=1)
            translation = my[valid, 0] - scale[:, None] * np.einsum("fij,fj->fi", R, mx[valid, 0])
            aligned[valid] = scale[:, None, None] * (X[valid] @ R.transpose(0, 2, 1)) + translation[:, None, :]

        errors[start:stop] = np.linalg.norm(aligned - Y, axis=-1)

    return errors


def mpjve(pred, gt_velocity_frame):
    '''MPJVE (mm/frame) between a pelvis-centred prediction and a pelvis-centred GT in the same frame.'''
    if pred.shape != gt_velocity_frame.shape:
        raise ValueError(f"Shape mismatch: {pred.shape} vs {gt_velocity_frame.shape}")

    v_pred = np.diff(pred.astype(np.float64), axis=0)
    v_gt = np.diff(gt_velocity_frame.astype(np.float64), axis=0)
    return float(np.linalg.norm(v_pred - v_gt, axis=-1).mean())


def list_sequences(model_name):
    root = MODEL_CONFIGS[model_name]["prediction_root"]
    if root is None:
        raise FileNotFoundError(f"No prediction folder found for {model_name}; run its inference notebook first.")

    items = []
    for subject in SUBJECTS:
        subject_dir = root / subject
        if not subject_dir.is_dir():
            raise FileNotFoundError(f"Missing directory: {subject_dir}")
        for name in sorted(os.listdir(subject_dir)):
            if name.endswith(".npy") and not is_warmup(name[:-4]):
                items.append((subject, name[:-4]))
    return items


def load_prediction(model_name, subject, video):
    cfg = MODEL_CONFIGS[model_name]
    path = cfg["prediction_root"] / subject / f"{video}.npy"
    pose = np.load(path, allow_pickle=False)

    if pose.ndim != 3 or pose.shape[1:] != (cfg["num_joints"], 3):
        raise ValueError(f"{path}: expected (T, {cfg['num_joints']}, 3), received {pose.shape}")
    if len(pose) == 0 or not np.isfinite(pose).all():
        raise ValueError(f"{path}: empty or non-finite prediction")

    return pose.astype(np.float32), path


print("Helper functions defined.")

### Table and figure helpers

The following cell defines the helpers that build the per-subject tables, write them as LaTeX and CSV, save the figures and compare every result with the value printed in the paper.

In [ ]:
# Helpers: tables, figures and comparison with the paper

CHECKS = []   # every comparison with the paper is recorded here and summarized in Section 7


def subject_table(df, value_col, row_keys, row_labels, subjects=SUBJECTS):
    '''Mean of `value_col` per subject and over all sequences (column "Mean") for each row.

    row_keys   : list of dicts {column: value} selecting the sequences of each row
    row_labels : label of each row
    '''
    rows = {}
    for key, label in zip(row_keys, row_labels):
        mask = np.ones(len(df), dtype=bool)
        for column, value in key.items():
            mask &= (df[column] == value).to_numpy()
        part = df[mask]

        if part.empty:
            continue

        row = part.groupby("subject")[value_col].mean().reindex(subjects).to_dict()
        row["Mean"] = float(part[value_col].mean())
        rows[label] = row

    return pd.DataFrame(rows).T[list(subjects) + ["Mean"]]


def latex_table(table, caption, label, decimals, row_header, group_size=None):
    '''LaTeX (booktabs) code of a per-subject table; the last column is the mean.'''
    def clean(text):
        return str(text).replace("–", "--")

    columns = list(table.columns)
    lines = [
        r"\begin{table}[!htbp]",
        r"\centering",
        r"\caption{" + clean(caption) + "}",
        r"\label{" + label + "}",
        r"\resizebox{\linewidth}{!}{%",
        r"\begin{tabular}{l" + "c" * len(columns) + "}",
        r"\toprule",
        row_header + " & " + " & ".join(columns[:-1]) + r" & \textbf{Mean} \\",
        r"\midrule",
    ]

    for index, (name, row) in enumerate(table.iterrows()):
        if group_size and index > 0 and index % group_size == 0:
            lines.append(r"\addlinespace")
        values = " & ".join(f"{row[c]:.{decimals}f}" for c in columns[:-1])
        lines.append(clean(name) + " & " + values + r" & \textbf{" + f"{row['Mean']:.{decimals}f}" + r"} \\")

    lines += [r"\bottomrule", r"\end{tabular}}", r"\end{table}"]
    return "\n".join(lines) + "\n"


def save_table(table, name, caption, label, decimals, row_header, group_size=None):
    '''Write a table as LaTeX and CSV into TABLES_DIR.'''
    tex_path = TABLES_DIR / f"{name}.tex"
    csv_path = TABLES_DIR / f"{name}.csv"
    tex_path.write_text(latex_table(table, caption, label, decimals, row_header, group_size), encoding="utf-8")
    table.to_csv(csv_path, index_label=row_header, float_format="%.6f")
    print(f"Saved: {tex_path.name} and {csv_path.name} in {TABLES_DIR}")


def show_table(table, decimals):
    try:
        display(table.style.format(f"{{:.{decimals}f}}"))
    except ImportError:          # the styler needs jinja2
        display(table.round(decimals))


def save_figure(fig, name):
    '''Save a figure as PNG and/or PDF into FIGURES_DIR.'''
    saved = []
    if SAVE_PNG:
        path = FIGURES_DIR / f"{name}.png"
        fig.savefig(path, dpi=PNG_DPI, bbox_inches="tight", pad_inches=0.03, facecolor="white")
        saved.append(path.name)
    if SAVE_PDF:
        path = FIGURES_DIR / f"{name}.pdf"
        fig.savefig(path, format="pdf", bbox_inches="tight", pad_inches=0.03, facecolor="white")
        saved.append(path.name)
    print("Saved:", ", ".join(saved), "in", FIGURES_DIR)


def compare_with_paper(table, paper, name, tolerance_fn):
    '''Compare a table with the paper values.

    paper        : {row label: [one value per subject..., mean]}
    tolerance_fn : function(row label) -> (subject tolerance, mean tolerance)
    The result is stored in CHECKS and a summary is printed.
    '''
    rows = []
    worst_subject, worst_mean = 0.0, 0.0
    status_ok = True

    for label, reference in paper.items():
        if label not in table.index:
            status_ok = False
            rows.append({"row": label, "max subject dev.": np.nan, "mean dev.": np.nan, "status": "MISSING"})
            continue

        reference = np.asarray(reference, dtype=float)
        computed = table.loc[label].to_numpy(dtype=float)
        subject_dev = float(np.nanmax(np.abs(computed[:-1] - reference[:-1])))
        mean_dev = float(abs(computed[-1] - reference[-1]))
        tol_subject, tol_mean = tolerance_fn(label)
        ok = subject_dev <= tol_subject and mean_dev <= tol_mean

        status_ok &= ok
        worst_subject = max(worst_subject, subject_dev)
        worst_mean = max(worst_mean, mean_dev)
        rows.append({
            "row": label,
            "max subject dev.": round(subject_dev, 4),
            "mean dev.": round(mean_dev, 4),
            "status": "OK" if ok else "CHECK",
        })

    display(pd.DataFrame(rows))
    CHECKS.append({
        "check": name,
        "max deviation (subjects)": worst_subject,
        "max deviation (mean)": worst_mean,
        "status": "OK" if status_ok else "CHECK",
    })
    print(f"{name}: {'OK' if status_ok else 'CHECK'} "
          f"(largest deviation: subjects {worst_subject:.4f}, mean {worst_mean:.4f})")


def grouped_barh(pivot, title, xlabel, name, label_size=12, left_margin=0.31, figure_width=12,
                 height_per_item=0.42, group_width=0.80, sort=True, colors=None, legend_names=None):
    '''Horizontal grouped bars: one group per row of `pivot`, one bar per column.'''
    pivot = pivot.copy()
    if sort:
        pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

    labels = pivot.index.tolist()
    n_items, n_series = len(labels), pivot.shape[1]
    positions = np.arange(n_items)
    bar_height = group_width / n_series

    fig, ax = plt.subplots(figsize=(figure_width, max(6, height_per_item * n_items)))

    for i, column in enumerate(pivot.columns):
        offset = (i - (n_series - 1) / 2) * bar_height
        ax.barh(
            positions + offset, pivot[column].to_numpy(), height=bar_height,
            label=(legend_names or {}).get(column, column),
            color=None if colors is None else colors[column],
        )

    ax.set_yticks(positions)
    ax.set_yticklabels(labels, fontsize=label_size)
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_title(title, fontsize=12)
    ax.tick_params(axis="x", labelsize=10)
    ax.grid(axis="x", alpha=0.3)
    ax.set_axisbelow(True)
    ax.legend(fontsize=10.5, loc="lower right")
    ax.invert_yaxis()
    fig.subplots_adjust(left=left_margin, right=0.97, top=0.94, bottom=0.10)

    save_figure(fig, name)
    plt.show()
    plt.close(fig)
    return pivot


print("Table and figure helpers defined.")

## 3. Influence of the 2D Detector (Table 4, Figs. 4 and 5)

TCPFormer and PoseMamba-L are evaluated with three sources of 2D keypoints: the projection of the ground-truth 3D joints (**GT 2D**), **YOLO-Pose** and **RTMPose**. Each combination is produced by running notebooks 01 and 02 with the corresponding `POSE_SOURCE`; every run writes `results_per_video_<source>_overlap_hann_w243_s60.csv` with the PA-MPJPE of every sequence on the original 17-joint skeleton.

| `POSE_SOURCE` | 2D source |
|---|---|
| `poses2d_gt` | GT 2D |
| `pose2d_yolo` | YOLO-Pose |
| `pose2d_rtmpose` | RTMPose |

This section reads the six CSV files, builds **Table 4** (mean PA-MPJPE per subject) and draws **Fig. 4** (TCPFormer) and **Fig. 5** (PoseMamba-L): the PA-MPJPE of every exercise for the three 2D sources, sorted from the highest to the lowest mean error.

In [ ]:
# Load the per-sequence results of the three 2D sources

# Source label -> names under which the CSV file may have been saved (the first one found is used;
# the last names of each list are kept only to read results produced with earlier versions of the notebooks).
SOURCES_2D = {
    "GT 2D": ["poses2d_gt", "pose2d_gt", "poses2d_oracle", "pose2d_oracle"],
    "YOLO-Pose": ["pose2d_yolo", "poses2d_yolo", "poses2d", "pose2d"],
    "RTMPose": ["pose2d_rtmpose", "poses2d_rtmpose"],
}
MONOCULAR_MODELS = ["TCPFormer", "PoseMamba-L"]


def find_source_csv(model_name, aliases):
    for root in MODEL_ROOTS[model_name]:
        for alias in aliases:
            path = root / f"results_per_video_{alias}_{RUN_TAG}.csv"
            if path.is_file():
                return path
    return None


frames_2d = []
missing_sources = []

for model_name in MONOCULAR_MODELS:
    for source, aliases in SOURCES_2D.items():
        path = find_source_csv(model_name, aliases)

        if path is None:
            missing_sources.append((model_name, source))
            print(f"MISSING | {model_name} | {source}: results_per_video_{aliases[0]}_{RUN_TAG}.csv not found")
            continue

        table = pd.read_csv(path)
        table["subject"] = table["subject"].astype(str).str.strip()
        table["video"] = table["video"].astype(str).str.strip()
        table = table[table["subject"].isin(SUBJECTS) & ~table["video"].map(is_warmup)].copy()

        if table.duplicated(["subject", "video"]).any():
            raise RuntimeError(f"{path}: duplicated sequences")

        table["model"] = model_name
        table["source"] = source
        frames_2d.append(table[["model", "source", "subject", "video", "PA_MPJPE_mm"]])

        flag = "" if len(table) == EXPECTED_SEQUENCES_PER_MODEL else f"   <-- expected {EXPECTED_SEQUENCES_PER_MODEL}"
        print(f"{model_name:12s} | {source:9s} | {len(table):3d} sequences | {path}{flag}")

if not frames_2d:
    raise FileNotFoundError("No per-sequence result file of notebooks 01 and 02 was found.")

df_2d = pd.concat(frames_2d, ignore_index=True)

if missing_sources:
    print(f"\n{len(missing_sources)} of 6 combinations are missing; the table and figures below show the available ones.")

In [ ]:
# Table 4: mean PA-MPJPE (mm) per subject, original 17-joint skeleton

keys, labels = [], []
for model_name in MONOCULAR_MODELS:
    for source in SOURCES_2D:
        keys.append({"model": model_name, "source": source})
        labels.append(f"{model_name} – {source}")

table4 = subject_table(df_2d, "PA_MPJPE_mm", keys, labels)
show_table(table4, 2)

save_table(
    table4, "table4_pa_mpjpe_2d_sources",
    caption=("Mean PA-MPJPE (mm) per subject for TCPFormer and PoseMamba-L using three 2D input sources, "
             "evaluated on their original 17-joint representation."),
    label="tab:pampjpe_2d_sources", decimals=2, row_header="Model / input", group_size=3,
)

# Values printed in the paper (Table 4): eight subjects and mean.
PAPER_TABLE4 = {
    "TCPFormer – GT 2D":         [42.17, 50.02, 42.02, 45.91, 52.15, 39.92, 49.56, 47.50, 46.16],
    "TCPFormer – YOLO-Pose":     [87.52, 94.19, 95.21, 103.65, 95.49, 90.16, 97.91, 84.71, 93.61],
    "TCPFormer – RTMPose":       [69.57, 74.63, 71.88, 70.94, 71.23, 71.96, 69.70, 67.20, 70.89],
    "PoseMamba-L – GT 2D":       [34.87, 45.79, 34.07, 36.76, 38.78, 31.30, 39.83, 35.57, 37.12],
    "PoseMamba-L – YOLO-Pose":   [78.68, 85.83, 82.51, 94.17, 85.16, 80.35, 92.54, 75.32, 84.32],
    "PoseMamba-L – RTMPose":     [71.55, 77.07, 69.36, 69.05, 71.75, 72.13, 69.43, 68.81, 71.14],
}

compare_with_paper(table4, PAPER_TABLE4, "Table 4 (2D sources, 17 joints)", lambda label: (0.15, 0.08))

In [ ]:
# Figs. 4 and 5: PA-MPJPE per exercise for each 2D source

FIGURE_NAMES_2D = {
    "TCPFormer": "fig04_tcpformer_pa_mpjpe_by_exercise",
    "PoseMamba-L": "fig05_posemamba_l_pa_mpjpe_by_exercise",
}

for model_name in MONOCULAR_MODELS:
    part = df_2d[df_2d["model"] == model_name]

    if part.empty:
        print(f"{model_name}: no data, figure skipped.")
        continue

    pivot = part.groupby(["video", "source"])["PA_MPJPE_mm"].mean().unstack("source")
    pivot = pivot[[source for source in SOURCES_2D if source in pivot.columns]]

    pivot = grouped_barh(
        pivot,
        title=f"{model_name} - Mean PA-MPJPE by exercise",
        xlabel="Mean PA-MPJPE (mm)",
        name=FIGURE_NAMES_2D[model_name],
    )

    pivot.to_csv(CSV_DIR / f"{FIGURE_NAMES_2D[model_name]}.csv", float_format="%.6f")

## 4. Monocular vs. Multi-view Comparison (Table 5, Figs. 6, 7 and 8)

The three models are compared on the **COMMON15** skeleton. TCPFormer and PoseMamba-L use the predictions obtained with **RTMPose** keypoints, and MVGFormer the predictions obtained from the four synchronized cameras. For each of the 224 sequences of each model:

1. the prediction and the ground truth are converted to COMMON15, in millimetres, and centred on the pelvis (Section 2);
2. every frame of the prediction is aligned to the ground truth with a similarity transformation, and the error of each of the 15 joints is averaged over the frames of the sequence.

The sequence-level PA-MPJPE is the mean of the 15 joint errors. Both the per-sequence and the per-joint results are saved to `Fit3D_evaluation/csv/`. The pelvis is the root of the pelvis-centred poses, but after the per-frame Procrustes alignment its error is not zero and it is included in the average.

In [ ]:
# PA-MPJPE on COMMON15 for every sequence and joint

joint_rows, video_rows = [], []

for model_name in MODEL_ORDER:
    cfg = MODEL_CONFIGS[model_name]
    sequences = list_sequences(model_name)

    if len(sequences) != EXPECTED_SEQUENCES_PER_MODEL:
        raise RuntimeError(f"{model_name}: expected {EXPECTED_SEQUENCES_PER_MODEL} sequences, found {len(sequences)}")

    print(f"{model_name}: {len(sequences)} sequences | {cfg['prediction_root']}")

    for index, (subject, video) in enumerate(sequences, start=1):
        pose, _ = load_prediction(model_name, subject, video)
        gt = load_gt_common15(subject, video, cfg["gt_swap"])

        n_frames = min(len(pose), len(gt))
        prediction = to_common15_mm(pose[:n_frames], cfg)

        errors = pa_errors(prediction, gt[:n_frames])         # (T, 15)
        per_joint = errors.mean(axis=0)

        video_rows.append({
            "model": model_name, "subject": subject, "video": video,
            "n_eval_frames": n_frames, "PA_MPJPE_COMMON15_mm": float(errors.mean()),
        })
        for joint_index, joint_name in enumerate(COMMON15_NAMES):
            joint_rows.append({
                "model": model_name, "subject": subject, "video": video,
                "joint": joint_name, "PA_MPJPE_mm": float(per_joint[joint_index]),
            })

        if index % 56 == 0 or index == len(sequences):
            print(f"  {index}/{len(sequences)} sequences evaluated")

df_common15 = pd.DataFrame(video_rows)
df_common15_joint = pd.DataFrame(joint_rows)

df_common15.to_csv(CSV_DIR / "COMMON15_PA_MPJPE_per_video.csv", index=False, float_format="%.9f")
df_common15_joint.to_csv(CSV_DIR / "COMMON15_PA_MPJPE_per_joint.csv", index=False, float_format="%.9f")

display(df_common15.groupby("model")["PA_MPJPE_COMMON15_mm"].agg(["count", "mean"]).reindex(MODEL_ORDER).round(3))

In [ ]:
# Table 5: mean PA-MPJPE (mm) per subject on COMMON15

LABELS_TABLE5 = {
    "TCPFormer": "TCPFormer – RTMPose",
    "PoseMamba-L": "PoseMamba-L – RTMPose",
    "MVGFormer": "MVGFormer (multi-view)",
}

table5 = subject_table(
    df_common15, "PA_MPJPE_COMMON15_mm",
    [{"model": model_name} for model_name in MODEL_ORDER],
    [LABELS_TABLE5[model_name] for model_name in MODEL_ORDER],
)
show_table(table5, 3)

save_table(
    table5, "table5_pa_mpjpe_common15",
    caption=("Mean PA-MPJPE (mm) per subject for TCPFormer, PoseMamba-L, and MVGFormer using the common 15-joint "
             "representation. The monocular models use RTMPose as 2D input."),
    label="tab:pampjpe", decimals=3, row_header="Model / input",
)

# Values printed in the paper (Table 5): eight subjects and mean.
PAPER_TABLE5 = {
    "TCPFormer – RTMPose":    [68.394, 71.662, 71.473, 70.798, 72.220, 70.441, 67.010, 64.527, 69.566],
    "PoseMamba-L – RTMPose":  [68.922, 74.660, 68.557, 68.321, 71.521, 69.209, 64.481, 64.476, 68.768],
    "MVGFormer (multi-view)": [61.680, 60.302, 67.724, 70.707, 77.148, 71.247, 55.809, 68.969, 66.698],
}

# MVGFormer (Original): 66.698 mm with the evaluation of this repository. The arXiv version of the paper lists
# 66.691 mm for the original predictions, computed with a Procrustes variant that also applies the reflection
# correction to the scale (difference below 0.02 mm per subject). A small numerical difference is also allowed
# when the multi-view inference is run again on another GPU or software version.
compare_with_paper(
    table5, PAPER_TABLE5, "Table 5 (COMMON15 PA-MPJPE)",
    lambda label: (0.15, 0.05) if label.startswith("MVGFormer") else (0.05, 0.02),
)

In [ ]:
# Fig. 6: PA-MPJPE per exercise (COMMON15)

CUSTOM_EXERCISE_LABELS = {
    "neutral_overhead_shoulder_press": "neutral overhead press",
    "dumbbell_overhead_shoulder_press": "DB overhead press",
}


def exercise_label(name):
    name = CUSTOM_EXERCISE_LABELS.get(name, name.replace("_", " "))
    return "\n".join(textwrap.wrap(name, width=27, break_long_words=False))


pivot6 = (
    df_common15.groupby(["video", "model"])["PA_MPJPE_COMMON15_mm"].mean()
    .unstack("model")
    .reindex(columns=MODEL_ORDER)
)

if pivot6.isna().any().any():
    raise RuntimeError("Some exercise has no result for one of the models.")

pivot6.to_csv(CSV_DIR / "fig06_comparison_pa_mpjpe_by_exercise_common15.csv", float_format="%.6f")
pivot6.index = [exercise_label(name) for name in pivot6.index]

_ = grouped_barh(
    pivot6,
    title="Model comparison by exercise - Mean PA-MPJPE",
    xlabel="Mean PA-MPJPE (mm)",
    name="fig06_comparison_pa_mpjpe_by_exercise",
)

In [ ]:
# Fig. 8: PA-MPJPE per joint (COMMON15)

MODEL_COLORS = {"TCPFormer": "#1f77b4", "PoseMamba-L": "#ff7f0e", "MVGFormer": "#2ca02c"}

# The mean error of each joint is first averaged over the frames of every sequence and then over the 224 sequences.
pivot8 = (
    df_common15_joint.groupby(["joint", "model"])["PA_MPJPE_mm"].mean()
    .unstack("model")
    .reindex(index=COMMON15_NAMES, columns=MODEL_ORDER)
)

if pivot8.isna().any().any():
    raise RuntimeError("Some joint has no result for one of the models.")

pivot8.to_csv(CSV_DIR / "fig08_comparison_pa_mpjpe_by_joint_common15.csv", index_label="Joint", float_format="%.6f")

fig, ax = plt.subplots(figsize=(14, 8.5))
positions = np.arange(len(COMMON15_NAMES))
bar_height = 0.26

for i, model_name in enumerate(MODEL_ORDER):
    offset = (i - (len(MODEL_ORDER) - 1) / 2) * bar_height
    ax.barh(positions + offset, pivot8[model_name].to_numpy(), height=bar_height,
            color=MODEL_COLORS[model_name], label=model_name, zorder=3)

ax.set_yticks(positions)
ax.set_yticklabels(COMMON15_NAMES, fontsize=14)
ax.invert_yaxis()
ax.set_xlabel("Mean PA-MPJPE (mm)", fontsize=13)
ax.set_title("Model comparison by joint - Mean PA-MPJPE", fontsize=17, pad=12)
ax.tick_params(axis="x", labelsize=12)
ax.grid(axis="x", linewidth=0.8, alpha=0.25, zorder=0)
ax.set_axisbelow(True)
ax.set_xlim(left=0, right=max(110, float(pivot8.to_numpy().max()) * 1.05))
ax.legend(loc="lower right", fontsize=12, frameon=True)
fig.tight_layout()

save_figure(fig, "fig08_comparison_pa_mpjpe_by_joint")
plt.show()
plt.close(fig)

display(pivot8.round(2))

### Qualitative comparison (Fig. 7)

Fig. 7 compares the skeletons of the three models with the ground truth for three frames of the *man maker* sequence of subject `s03`. In the last two frames the person lies on the floor.

The poses are drawn in the COMMON15 skeleton. Because the 3D predictions are expressed in different coordinate frames and scales (the monocular models in the camera frame, MVGFormer in the world frame), each prediction is aligned to the ground truth with **one** similarity transformation estimated from the whole sequence (this is not the frame-wise Procrustes alignment of the metric). The same view and the same axis limits are used for the four skeletons of a frame. The left/right convention of each model is corrected on the predictions (equivalent to correcting the ground truth).

In [ ]:
# Skeleton drawing helpers (Figs. 7 and 11)

def similarity_transform(X, Y):
    '''Similarity transformation (rotation, translation and uniform scale) that best maps the points X onto Y.'''
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)

    mu_x, mu_y = X.mean(axis=0), Y.mean(axis=0)
    X0, Y0 = X - mu_x, Y - mu_y
    norm_x, norm_y = np.linalg.norm(X0), np.linalg.norm(Y0)

    if norm_x < 1e-12 or norm_y < 1e-12:
        return X + (mu_y - mu_x)

    U, s, Vt = np.linalg.svd((X0 / norm_x).T @ (Y0 / norm_y))
    R = Vt.T @ U.T

    if np.linalg.det(R) < 0:
        Vt[-1] *= -1
        s[-1] *= -1
        R = Vt.T @ U.T

    scale = norm_y / norm_x * s.sum()
    translation = mu_y - scale * (R @ mu_x)
    return scale * (R @ X.T).T + translation


def global_sequence_pa(pred, gt):
    '''Align a whole sequence (T, J, 3) to the ground truth with a single similarity transformation.'''
    aligned = similarity_transform(pred.reshape(-1, 3), gt.reshape(-1, 3))
    return aligned.reshape(pred.shape).astype(np.float32)


def aligned_for_display(model_name, subject, video, arrays):
    '''Ground truth (original left/right labels) and globally aligned COMMON15 versions of a model.

    arrays : {label: stored-units prediction (T, J, 3)}
    '''
    cfg = MODEL_CONFIGS[model_name]
    gt = load_gt_common15(subject, video, None)
    n_frames = min([len(gt)] + [len(a) for a in arrays.values()])
    gt = gt[:n_frames]

    aligned = {}
    for label, array in arrays.items():
        prediction = swap_sides(to_common15_mm(array[:n_frames], cfg), cfg["gt_swap"])
        aligned[label] = global_sequence_pa(prediction, gt)

    return gt, aligned


def common_limits(poses, margin):
    '''Cubic axis limits shared by several poses (J, 3) of the same row.'''
    points = np.concatenate([np.asarray(p).reshape(-1, 3) for p in poses], axis=0)
    center = np.nanmean(points, axis=0)
    ranges = np.nanpercentile(points, 99, axis=0) - np.nanpercentile(points, 1, axis=0)
    half = float(np.nanmax(ranges)) * margin

    if not np.isfinite(half) or half <= 0:
        half = 500.0

    return [(center[k] - half, center[k] + half) for k in range(3)]


def draw_skeleton(ax, pose, limits, azimuth, elevation=12, line_width=3.8, point_size=34):
    '''Draw one COMMON15 pose (J, 3) on a 3D axis.'''
    ax.set_xlim(limits[0])
    ax.set_ylim(limits[1])
    ax.set_zlim(limits[2])
    ax.set_box_aspect([1, 1, 1])
    ax.set_proj_type("ortho")
    ax.view_init(elev=elevation, azim=azimuth)
    ax.set_axis_off()

    x, y, z = pose[:, 0], pose[:, 1], pose[:, 2]

    for k, (i, j) in enumerate(SKELETON_15):
        ax.plot([x[i], x[j]], [y[i], y[j]], [z[i], z[j]],
                color=f"C{k % 10}", linewidth=line_width, solid_capstyle="round")

    ax.scatter(x, y, z, s=point_size, color="C0")


def plot_skeleton_grid(column_titles, rows, name, margin, cell_size=2.3, line_width=3.8, point_size=34):
    '''Grid of skeletons: one row per entry of `rows` ({"label", "poses", "azimuth"}), one column per title.'''
    n_rows, n_cols = len(rows), len(column_titles)
    fig = plt.figure(figsize=(cell_size * n_cols + 0.6, cell_size * n_rows + 0.5))

    for r, row in enumerate(rows):
        limits = common_limits(row["poses"], margin)

        for c, pose in enumerate(row["poses"]):
            ax = fig.add_subplot(n_rows, n_cols, r * n_cols + c + 1, projection="3d")
            draw_skeleton(ax, pose, limits, row["azimuth"], line_width=line_width, point_size=point_size)

            if r == 0:
                ax.set_title(column_titles[c], fontsize=12, fontweight="bold")
            if c == 0:
                ax.text2D(-0.02, 0.5, row["label"], transform=ax.transAxes, rotation=90,
                          ha="right", va="center", fontsize=11)

    fig.subplots_adjust(left=0.07, right=0.995, top=0.93, bottom=0.0, wspace=0.0, hspace=0.0)
    save_figure(fig, name)
    plt.show()
    plt.close(fig)


print("Skeleton helpers defined.")

In [ ]:
# Fig. 7: GT, TCPFormer, PoseMamba-L and MVGFormer on three frames of s03 / man_maker

FIG7_SUBJECT = "s03"
FIG7_VIDEO = "man_maker"
FIG7_FRAMES = [568, 1854, 1958]       # frame indices (0-based)
FIG7_AZIMUTH = 0                      # fixed viewing azimuth (elevation: 12 degrees)

ground_truth, aligned_models = None, {}

for model_name in MODEL_ORDER:
    pose, _ = load_prediction(model_name, FIG7_SUBJECT, FIG7_VIDEO)
    gt_model, aligned = aligned_for_display(model_name, FIG7_SUBJECT, FIG7_VIDEO, {model_name: pose})
    aligned_models[model_name] = aligned[model_name]
    ground_truth = gt_model if ground_truth is None else ground_truth

n_common = min([len(ground_truth)] + [len(a) for a in aligned_models.values()])
print(f"{FIG7_SUBJECT}/{FIG7_VIDEO}: {n_common} common frames")

if max(FIG7_FRAMES) >= n_common:
    raise ValueError(f"Frame {max(FIG7_FRAMES)} is not available (the sequence has {n_common} frames).")

rows = [{
    "label": f"Frame {frame}",
    "poses": [ground_truth[frame]] + [aligned_models[m][frame] for m in MODEL_ORDER],
    "azimuth": FIG7_AZIMUTH,
} for frame in FIG7_FRAMES]

plot_skeleton_grid(["Ground truth"] + MODEL_ORDER, rows, "fig07_man_maker_qualitative_comparison", margin=0.60)

## 5. Temporal Filtering (Tables 6 and 7, Figs. 9, 10 and 11)

Notebook 04 applies three temporal filters (One Euro, Savitzky–Golay and Median) to the predictions of each model and evaluates every version of every sequence on COMMON15. Its table `smoothing_COMMON15_per_video.csv` contains the PA-MPJPE and the MPJVE of the 224 sequences for the original and the three filtered versions of each model (2,688 rows).

This section builds from that file **Table 6** (PA-MPJPE) and **Table 7** (MPJVE), and verifies that the original predictions evaluated in Section 4 give the same PA-MPJPE as in notebook 04.

In [ ]:
# Load the evaluation of the filtered predictions (notebook 04)

if not SMOOTHING_METRICS_CSV.is_file():
    raise FileNotFoundError(
        f"{SMOOTHING_METRICS_CSV} not found. Run notebook 04 up to its verification section "
        "(Section 8), which writes this file."
    )

FILTER_KEYS_ALL = ["original", "one_euro", "savgol", "median"]

FILTER_LABELS = {
    "original": "Original",
    "one_euro": "One Euro filter",
    "savgol": "Savitzky–Golay filter",
    "median": "Median filter",
}

df_smoothing = pd.read_csv(SMOOTHING_METRICS_CSV)

required_columns = {"model", "filter", "subject", "video", "PA_MPJPE_COMMON15_mm", "MPJVE_mm_frame"}
if not required_columns.issubset(df_smoothing.columns):
    raise ValueError(f"Missing columns in {SMOOTHING_METRICS_CSV.name}: {sorted(required_columns - set(df_smoothing.columns))}")

df_smoothing["subject"] = df_smoothing["subject"].astype(str).str.strip()
df_smoothing["video"] = df_smoothing["video"].astype(str).str.strip()
df_smoothing = df_smoothing[df_smoothing["subject"].isin(SUBJECTS) & ~df_smoothing["video"].map(is_warmup)].copy()

counts = df_smoothing.groupby(["model", "filter"]).size().unstack("filter").reindex(index=MODEL_ORDER, columns=FILTER_KEYS_ALL)
display(counts)

if not (counts.to_numpy() == EXPECTED_SEQUENCES_PER_MODEL).all():
    raise RuntimeError(f"Every model/filter combination must contain {EXPECTED_SEQUENCES_PER_MODEL} sequences.")

if df_smoothing.duplicated(["model", "filter", "subject", "video"]).any():
    raise RuntimeError("Duplicated sequences in the smoothing metrics.")

# The original predictions must give the same PA-MPJPE here (Section 4) and in notebook 04.
original_04 = df_smoothing[df_smoothing["filter"] == "original"][["model", "subject", "video", "PA_MPJPE_COMMON15_mm"]]
merged = df_common15.merge(original_04, on=["model", "subject", "video"], suffixes=("", "_04"), how="outer", validate="one_to_one", indicator=True)

if not (merged["_merge"] == "both").all():
    raise RuntimeError("Sections 4 and notebook 04 do not contain the same sequences.")

difference = float((merged["PA_MPJPE_COMMON15_mm"] - merged["PA_MPJPE_COMMON15_mm_04"]).abs().max())
CHECKS.append({
    "check": "Original PA-MPJPE: this notebook vs. notebook 04",
    "max deviation (subjects)": difference,
    "max deviation (mean)": difference,
    "status": "OK" if difference < 1e-3 else "CHECK",
})
print(f"Maximum difference of the PA-MPJPE of the original predictions: {difference:.2e} mm")

In [ ]:
# Table 6 (PA-MPJPE) and Table 7 (MPJVE) per subject

keys, labels = [], []
for model_name in MODEL_ORDER:
    for family in FILTER_KEYS_ALL:
        keys.append({"model": model_name, "filter": family})
        labels.append(f"{model_name} – {FILTER_LABELS[family]}")

table6 = subject_table(df_smoothing, "PA_MPJPE_COMMON15_mm", keys, labels)
table7 = subject_table(df_smoothing, "MPJVE_mm_frame", keys, labels)

print("Table 6: mean PA-MPJPE (mm)")
show_table(table6, 3)
print("Table 7: mean MPJVE (mm/frame)")
show_table(table7, 3)

save_table(
    table6, "table6_pa_mpjpe_filtering",
    caption=("Comparison of mean PA-MPJPE (mm) per subject and model, before and after temporal filtering, "
             "evaluated on the common 15-joint skeleton (COMMON15). TCPFormer and PoseMamba-L use the RTMPose "
             "detections as 2D input."),
    label="tab:pampjpe_filtering", decimals=3, row_header="Model / version", group_size=4,
)
save_table(
    table7, "table7_mpjve_filtering",
    caption=("Mean Per-Joint Velocity Error (MPJVE, mm/frame) with respect to the ground truth per subject, comparing "
             "the original predictions with the One Euro, Savitzky–Golay and Median filters, evaluated on the common "
             "15-joint skeleton. TCPFormer and PoseMamba-L use RTMPose as 2D input."),
    label="tab:mpjve_filtering", decimals=3, row_header="Model / version", group_size=4,
)

In [ ]:
# Comparison of Tables 6 and 7 with the paper

PAPER_TABLE6 = {
    "TCPFormer – Original":                [68.394, 71.662, 71.473, 70.798, 72.220, 70.441, 67.010, 64.527, 69.566],
    "TCPFormer – One Euro filter":         [68.560, 71.779, 71.621, 70.897, 72.427, 70.552, 67.194, 64.679, 69.713],
    "TCPFormer – Savitzky–Golay filter":   [68.386, 71.651, 71.460, 70.787, 72.212, 70.431, 67.001, 64.511, 69.555],
    "TCPFormer – Median filter":           [68.394, 71.661, 71.470, 70.794, 72.220, 70.439, 67.008, 64.517, 69.563],
    "PoseMamba-L – Original":              [68.922, 74.660, 68.557, 68.321, 71.521, 69.209, 64.481, 64.476, 68.768],
    "PoseMamba-L – One Euro filter":       [68.923, 74.656, 68.555, 68.318, 71.526, 69.207, 64.485, 64.478, 68.769],
    "PoseMamba-L – Savitzky–Golay filter": [68.922, 74.654, 68.557, 68.321, 71.521, 69.209, 64.481, 64.475, 68.767],
    "PoseMamba-L – Median filter":         [68.923, 74.654, 68.557, 68.321, 71.521, 69.209, 64.481, 64.475, 68.768],
    "MVGFormer – Original":                [61.680, 60.302, 67.724, 70.707, 77.148, 71.247, 55.809, 68.969, 66.698],
    "MVGFormer – One Euro filter":         [63.150, 60.748, 67.726, 70.766, 76.539, 71.101, 58.285, 68.415, 67.091],
    "MVGFormer – Savitzky–Golay filter":   [56.937, 56.006, 62.148, 66.700, 72.272, 66.835, 52.274, 63.644, 62.102],
    "MVGFormer – Median filter":           [55.898, 55.140, 59.776, 65.407, 70.535, 63.953, 50.397, 61.973, 60.385],
}

PAPER_TABLE7 = {
    "TCPFormer – Original":                [2.823, 2.867, 3.216, 3.047, 3.325, 2.685, 2.925, 2.659, 2.943],
    "TCPFormer – One Euro filter":         [2.803, 2.848, 3.187, 3.011, 3.323, 2.632, 2.887, 2.608, 2.913],
    "TCPFormer – Savitzky–Golay filter":   [2.764, 2.825, 3.149, 2.972, 3.285, 2.592, 2.847, 2.557, 2.874],
    "TCPFormer – Median filter":           [2.792, 2.848, 3.184, 3.007, 3.308, 2.627, 2.880, 2.593, 2.905],
    "PoseMamba-L – Original":              [2.774, 2.824, 3.136, 2.938, 3.240, 2.578, 2.841, 2.539, 2.859],
    "PoseMamba-L – One Euro filter":       [2.775, 2.824, 3.137, 2.938, 3.241, 2.578, 2.841, 2.539, 2.859],
    "PoseMamba-L – Savitzky–Golay filter": [2.771, 2.821, 3.133, 2.936, 3.238, 2.575, 2.838, 2.536, 2.856],
    "PoseMamba-L – Median filter":         [2.774, 2.823, 3.136, 2.938, 3.240, 2.578, 2.840, 2.538, 2.858],
    "MVGFormer – Original":                [26.014, 23.492, 28.963, 26.565, 27.800, 27.443, 23.059, 28.332, 26.458],
    "MVGFormer – One Euro filter":         [6.646, 5.769, 7.449, 6.277, 7.081, 6.521, 5.552, 7.103, 6.550],
    "MVGFormer – Savitzky–Golay filter":   [4.696, 4.407, 5.393, 4.910, 5.509, 4.872, 4.228, 5.228, 4.905],
    "MVGFormer – Median filter":           [4.056, 3.860, 4.642, 4.326, 4.708, 3.950, 3.881, 4.116, 4.192],
}

compare_with_paper(
    table6, PAPER_TABLE6, "Table 6 (PA-MPJPE with filters)",
    lambda label: (0.15, 0.05) if label.startswith("MVGFormer") else (0.05, 0.02),
)
compare_with_paper(
    table7, PAPER_TABLE7, "Table 7 (MPJVE with filters)",
    lambda label: (0.10, 0.05) if label.startswith("MVGFormer") else (0.02, 0.01),
)

# Relative MPJVE reductions quoted in the text of the paper
def relative_reduction(model_name, family):
    original = table7.loc[f"{model_name} – {FILTER_LABELS['original']}", "Mean"]
    filtered = table7.loc[f"{model_name} – {FILTER_LABELS[family]}", "Mean"]
    return 100.0 * (1.0 - filtered / original)

for model_name, family, paper_value in [("TCPFormer", "savgol", 2.34), ("MVGFormer", "median", 84.2)]:
    value = relative_reduction(model_name, family)
    ok = abs(value - paper_value) <= 0.1
    CHECKS.append({
        "check": f"MPJVE reduction, {model_name} – {FILTER_LABELS[family]} (paper: {paper_value}%)",
        "max deviation (subjects)": np.nan,
        "max deviation (mean)": abs(value - paper_value),
        "status": "OK" if ok else "CHECK",
    })
    print(f"MPJVE reduction of {model_name} with {FILTER_LABELS[family]}: {value:.2f}% (paper: {paper_value}%)  {'OK' if ok else 'CHECK'}")

### Velocity curves (Figs. 9 and 10)

Figs. 9 and 10 show, for the sequence `band_pull_apart` of subject `s03`, the **mean body velocity** of the ground truth, of the original prediction and of the three filtered versions: the mean over the 15 joints of the displacement between consecutive frames, in mm/frame, on the COMMON15 skeleton. The upper plot shows the full sequence and the lower plot a zoom on frames 200 to 400. Fig. 9 corresponds to TCPFormer and Fig. 10 to MVGFormer.

The curves are a visualization of the temporal behaviour; the quantitative metric is the MPJVE of Table 7. A single scale factor, estimated from the original prediction as the ratio between the mean bone length of the ground truth and of the prediction, is applied to the original and to the three filtered versions, so that the size of the predicted body matches the ground truth without altering the effect of the filters. The scale is not used in the MPJVE.

The filtered predictions are read from the files written by notebook 04; their folder names encode the filter parameters, so the parameters below must be those used in notebook 04.

In [ ]:
# Final filter parameters (identical to notebook 04) and location of the filtered predictions

FINAL_FILTER_PARAMS = {
    "TCPFormer": {
        "one_euro": {"freq": 50.0, "min_cutoff": 10.0, "beta": 0.01, "d_cutoff": 2.0},
        "savgol":   {"window_length": 21, "polyorder": 4},
        "median":   {"window_size": 5},
    },
    "PoseMamba-L": {
        "one_euro": {"freq": 50.0, "min_cutoff": 100.0, "beta": 0.01, "d_cutoff": 1.0},
        "savgol":   {"window_length": 11, "polyorder": 4},
        "median":   {"window_size": 3},
    },
    "MVGFormer": {
        "one_euro": {"freq": 50.0, "min_cutoff": 1.0, "beta": 0.001, "d_cutoff": 1.0},
        "savgol":   {"window_length": 21, "polyorder": 2},
        "median":   {"window_size": 41},
    },
}


def parameter_tag(value):
    if isinstance(value, float):
        value = f"{value:g}"
    return str(value).replace(".", "p")


def filter_folder_name(family, params):
    if family == "one_euro":
        return (
            f"one_euro_freq{parameter_tag(params['freq'])}"
            f"_min{parameter_tag(params['min_cutoff'])}"
            f"_beta{parameter_tag(params['beta'])}"
            f"_d{parameter_tag(params['d_cutoff'])}"
        )
    if family == "savgol":
        return f"savgol_w{params['window_length']}_p{params['polyorder']}"
    if family == "median":
        return f"median_w{params['window_size']}"
    raise ValueError(family)


def filtered_path(model_name, family, subject, video):
    folder = filter_folder_name(family, FINAL_FILTER_PARAMS[model_name][family])
    return FILTERED_ROOT / model_name / folder / subject / f"{video}.npy"


def load_versions(model_name, subject, video):
    '''Original and filtered predictions of a sequence, in their stored units.'''
    versions = {"Original": load_prediction(model_name, subject, video)[0]}

    for family in ["one_euro", "savgol", "median"]:
        path = filtered_path(model_name, family, subject, video)
        if not path.is_file():
            raise FileNotFoundError(f"Filtered prediction not found: {path}. Run notebook 04 first.")
        versions[FILTER_LABELS[family]] = np.load(path, allow_pickle=False)

    return versions


print("Filtered predictions:", FILTERED_ROOT)

In [ ]:
# Figs. 9 and 10: mean body velocity of the original and filtered predictions

VELOCITY_SUBJECT = "s03"
VELOCITY_VIDEO = "band_pull_apart"
ZOOM_FRAMES = (200, 400)

# model -> (figure number, text appended to the plot title)
VELOCITY_MODELS = {"TCPFormer": (9, ", COMMON15"), "MVGFormer": (10, "")}

VELOCITY_ORDER = ["Ground truth", "Original", "One Euro", "Savitzky-Golay", "Median Filter"]
VERSION_TO_SIGNAL = {
    "Original": "Original",
    "One Euro filter": "One Euro",
    "Savitzky–Golay filter": "Savitzky-Golay",
    "Median filter": "Median Filter",
}
SIGNAL_TO_FILTER = {"Original": "original", "One Euro": "one_euro", "Savitzky-Golay": "savgol", "Median Filter": "median"}


def mean_bone_length(poses):
    lengths = [np.linalg.norm(poses[:, a, :].astype(np.float64) - poses[:, b, :].astype(np.float64), axis=-1) for a, b in SKELETON_15]
    return float(np.mean(lengths))


def mean_body_velocity(pose):
    '''Mean over joints of the displacement between consecutive frames (mm/frame).'''
    return np.linalg.norm(np.diff(pose.astype(np.float64), axis=0), axis=-1).mean(axis=1)


def velocity_case(model_name, subject, video):
    cfg = MODEL_CONFIGS[model_name]
    versions = load_versions(model_name, subject, video)
    gt = load_gt_common15(subject, video, cfg["gt_swap"])

    n_frames = min([len(gt)] + [len(v) for v in versions.values()])
    gt = gt[:n_frames]
    prepared = {label: to_common15_mm(array[:n_frames], cfg) for label, array in versions.items()}

    # One scale for all versions, estimated from the original prediction only.
    scale = mean_bone_length(gt) / mean_bone_length(prepared["Original"])

    signals = {"Ground truth": mean_body_velocity(gt)}
    for version, pose in prepared.items():
        signals[VERSION_TO_SIGNAL[version]] = mean_body_velocity(pose * scale)

    # MPJVE of the plotted sequence (no scaling), to be compared with Table 7 / notebook 04.
    if cfg["gt_frame"] == "camera":
        gt_velocity_frame = (gt.astype(np.float64) @ camera_rotation(subject, video).T).astype(np.float32)
    else:
        gt_velocity_frame = gt

    mpjve_values = {VERSION_TO_SIGNAL[version]: mpjve(pose, gt_velocity_frame) for version, pose in prepared.items()}

    return {"signals": signals, "scale": scale, "n_frames": n_frames, "mpjve": mpjve_values}


def plot_velocity(signals, model_name, subject, video, title_tag, name, zoom=None):
    n = min(len(v) for v in signals.values())
    frames = np.arange(1, n + 1)       # signal[k] is the displacement from frame k to k + 1 (1-based)

    if zoom is None:
        mask = np.ones(n, dtype=bool)
        title = f"{model_name} - {subject} - {video.replace('_', ' ')} (full sequence{title_tag})"
    else:
        mask = (frames >= zoom[0]) & (frames <= zoom[1])
        title = f"{model_name} - {subject} - {video.replace('_', ' ')} (frames {zoom[0]}--{zoom[1]}{title_tag})"

    fig = plt.figure(figsize=(12, 5))

    for label in VELOCITY_ORDER:
        values = signals[label][:n][mask]
        if label == "Ground truth":
            plt.plot(frames[mask], values, label=label, linewidth=2.4)
        elif label == "Original":
            plt.plot(frames[mask], values, label=label, linewidth=1.2, alpha=0.85)
        else:
            plt.plot(frames[mask], values, label=label, linewidth=1.7)

    if zoom is not None:
        values = np.concatenate([signals[label][:n][mask] for label in VELOCITY_ORDER])
        low, high = np.nanpercentile(values, 1), np.nanpercentile(values, 99)
        if np.isfinite(low) and np.isfinite(high) and high > low:
            margin = 0.15 * (high - low)
            plt.ylim(max(0, low - margin), high + margin)

    plt.title(title)
    plt.xlabel("Frame")
    plt.ylabel("Mean body velocity (mm/frame)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    save_figure(fig, name)
    plt.show()
    plt.close(fig)


velocity_checks = []

for model_name, (figure_number, title_tag) in VELOCITY_MODELS.items():
    case = velocity_case(model_name, VELOCITY_SUBJECT, VELOCITY_VIDEO)
    print(f"\n{model_name} | {VELOCITY_SUBJECT} | {VELOCITY_VIDEO} | {case['n_frames']} frames | body scale {case['scale']:.6f}")

    # Consistency of the plotted sequence with the MPJVE computed in notebook 04
    rows = []
    for signal_name, value in case["mpjve"].items():
        reference = df_smoothing[
            (df_smoothing["model"] == model_name) & (df_smoothing["filter"] == SIGNAL_TO_FILTER[signal_name])
            & (df_smoothing["subject"] == VELOCITY_SUBJECT) & (df_smoothing["video"] == VELOCITY_VIDEO)
        ]["MPJVE_mm_frame"].iloc[0]
        velocity_checks.append(abs(value - reference))
        rows.append({"version": signal_name, "MPJVE (mm/frame)": round(value, 4), "notebook 04": round(float(reference), 4)})
    display(pd.DataFrame(rows))

    signal_table = pd.DataFrame({"frame": np.arange(1, case["n_frames"]), **{k: v[: case["n_frames"] - 1] for k, v in case["signals"].items()}})
    signal_table.to_csv(CSV_DIR / f"fig{figure_number:02d}_{model_name.lower()}_{VELOCITY_SUBJECT}_{VELOCITY_VIDEO}_velocity_signals.csv", index=False, float_format="%.6f")

    base = f"fig{figure_number:02d}_{model_name.lower()}_velocity_{VELOCITY_SUBJECT}_{VELOCITY_VIDEO}"
    plot_velocity(case["signals"], model_name, VELOCITY_SUBJECT, VELOCITY_VIDEO, title_tag, base + "_full")
    plot_velocity(case["signals"], model_name, VELOCITY_SUBJECT, VELOCITY_VIDEO, title_tag, base + f"_frames_{ZOOM_FRAMES[0]}_{ZOOM_FRAMES[1]}", zoom=ZOOM_FRAMES)

deviation = float(max(velocity_checks))
CHECKS.append({
    "check": "Velocity figures: MPJVE of the plotted sequences vs. notebook 04",
    "max deviation (subjects)": deviation,
    "max deviation (mean)": deviation,
    "status": "OK" if deviation < 1e-3 else "CHECK",
})

### Qualitative effect of the filters on MVGFormer (Fig. 11)

Fig. 11 shows MVGFormer poses before and after filtering for two frames of subject `s03`: frame 142 of `band_pull_apart` (regular standing motion, with an isolated reconstruction error of one arm) and frame 2007 of `man_maker` (floor-supported pose). Each version is aligned to the ground truth with one similarity transformation per sequence, as in Fig. 7.

`FIG11_VIEW = "video"` uses the viewing azimuth that the frame has in a video in which the camera rotates 360° around the subject during the sequence; `"fixed"` uses `FIG11_AZIMUTH` for all frames.

In [ ]:
# Fig. 11: MVGFormer before and after temporal filtering

FIG11_SUBJECT = "s03"
FIG11_CASES = [("band_pull_apart", 142), ("man_maker", 2007)]     # (exercise, 0-based frame index)
FIG11_VIEW = "video"             # "video" or "fixed"
FIG11_AZIMUTH = 0                # used when FIG11_VIEW == "fixed"

rows = []

for video, frame in FIG11_CASES:
    versions = load_versions("MVGFormer", FIG11_SUBJECT, video)
    gt, aligned = aligned_for_display("MVGFormer", FIG11_SUBJECT, video, versions)
    n_frames = len(gt)

    if frame >= n_frames:
        raise ValueError(f"Frame {frame} is not available in {video} ({n_frames} frames).")

    azimuth = 360.0 * frame / (n_frames - 1) if FIG11_VIEW == "video" else FIG11_AZIMUTH

    rows.append({
        "label": f"{video.replace('_', ' ')}\nFrame {frame}",
        "poses": [gt[frame]] + [aligned[label][frame] for label in ["Original"] + [FILTER_LABELS[f] for f in ["one_euro", "savgol", "median"]]],
        "azimuth": azimuth,
    })

plot_skeleton_grid(
    ["Ground truth", "MVGFormer", "One Euro", "Savitzky–Golay", "Median Filter"],
    rows, "fig11_mvgformer_filters_qualitative", margin=0.52, cell_size=2.3, line_width=4.5, point_size=38,
)

## 6. Skeleton Conventions (Fig. 2)

Fig. 2 shows the three skeleton representations used in the experiments: (a) COCO-17, the output of YOLO-Pose and RTMPose; (b) the 17-joint Human3.6M representation used by TCPFormer and PoseMamba-L, which contains the separate `Nose` and `Head` joints; and (c) COMMON15, used for the comparison with MVGFormer, which retains `Nose` and has no `Head` joint. The figure does not depend on any result.

In [ ]:
# Fig. 2: skeleton conventions

LEFT_ARM, RIGHT_ARM = "#E83EAF", "#00B8C4"
LEFT_LEG, RIGHT_LEG = "#00A020", "#0072B2"
TORSO, HEAD = "#8A55C2", "#8C564B"
FACE_LEFT, FACE_RIGHT = "#F28E2B", "#009E73"
CENTER = "#707070"
JOINT_COLOR = "#0877A8"


def draw_skeleton_diagram(ax, joints, edges, labels, offsets, alignments, caption):
    for j1, j2, color in edges:
        (x1, y1), (x2, y2) = joints[j1], joints[j2]
        ax.plot([x1, x2], [y1, y2], color=color, linewidth=3.0, solid_capstyle="round", zorder=1)

    xs = [p[0] for p in joints.values()]
    ys = [p[1] for p in joints.values()]
    ax.scatter(xs, ys, s=58, color=JOINT_COLOR, edgecolor="white", linewidth=0.7, zorder=3)

    for joint_id, (x, y) in joints.items():
        dx, dy = offsets.get(joint_id, (0.17, 0.0))
        ha = alignments.get(joint_id, "left" if dx > 0 else "right" if dx < 0 else "center")
        ax.annotate(f"{joint_id}. {labels[joint_id]}", xy=(x, y), xytext=(x + dx, y + dy),
                    fontsize=11.5, ha=ha, va="center", zorder=4)

    width, height = max(xs) - min(xs), max(ys) - min(ys)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_xlim(min(xs) - width * 0.72, max(xs) + width * 0.72)
    ax.set_ylim(min(ys) - height * 0.10, max(ys) + height * 0.10)
    ax.text(0.5, -0.01, caption, transform=ax.transAxes, ha="center", va="top", fontsize=12.5)


# (a) COCO-17: YOLO-Pose and RTMPose
coco_joints = {
    0: (0.00, 9.00), 1: (-0.32, 9.35), 2: (0.32, 9.35), 3: (-0.68, 9.15), 4: (0.68, 9.15),
    5: (-1.25, 7.30), 6: (1.25, 7.30), 7: (-1.45, 5.80), 8: (1.45, 5.80), 9: (-1.20, 4.25), 10: (1.20, 4.25),
    11: (-0.70, 4.15), 12: (0.70, 4.15), 13: (-0.75, 2.10), 14: (0.75, 2.10), 15: (-0.75, 0.10), 16: (0.75, 0.10),
}
coco_labels = {
    0: "Nose", 1: "Left eye", 2: "Right eye", 3: "Left ear", 4: "Right ear", 5: "Left shoulder", 6: "Right shoulder",
    7: "Left elbow", 8: "Right elbow", 9: "Left wrist", 10: "Right wrist", 11: "Left hip", 12: "Right hip",
    13: "Left knee", 14: "Right knee", 15: "Left ankle", 16: "Right ankle",
}
coco_edges = [
    (1, 2, CENTER), (0, 1, FACE_LEFT), (0, 2, FACE_RIGHT), (1, 3, FACE_LEFT), (2, 4, FACE_RIGHT),
    (3, 5, FACE_LEFT), (4, 6, FACE_RIGHT), (5, 6, CENTER), (5, 7, LEFT_ARM), (7, 9, LEFT_ARM),
    (6, 8, RIGHT_ARM), (8, 10, RIGHT_ARM), (5, 11, LEFT_ARM), (6, 12, RIGHT_ARM), (11, 12, CENTER),
    (11, 13, LEFT_LEG), (13, 15, LEFT_LEG), (12, 14, RIGHT_LEG), (14, 16, RIGHT_LEG),
]
coco_offsets = {
    0: (0.00, -0.28), 1: (-0.12, 0.32), 2: (0.12, 0.32), 3: (-0.20, 0.00), 4: (0.20, 0.00),
    5: (-0.22, 0.00), 6: (0.22, 0.00), 7: (-0.22, 0.00), 8: (0.22, 0.00), 9: (-0.22, 0.00), 10: (0.22, 0.00),
    11: (-0.22, -0.32), 12: (0.22, -0.32), 13: (-0.22, 0.00), 14: (0.22, 0.00), 15: (-0.22, 0.00), 16: (0.22, 0.00),
}

# (b) 17-joint Human3.6M layout: TCPFormer and PoseMamba-L
h36m_joints = {
    0: (0.00, 4.20), 1: (0.70, 3.90), 2: (0.80, 2.10), 3: (0.75, 0.10), 4: (-0.70, 3.90), 5: (-0.80, 2.10), 6: (-0.75, 0.10),
    7: (0.00, 5.35), 8: (0.00, 6.65), 9: (0.00, 7.80), 10: (0.00, 9.00),
    11: (-1.25, 6.45), 12: (-1.55, 5.25), 13: (-1.70, 4.05), 14: (1.25, 6.45), 15: (1.55, 5.25), 16: (1.70, 4.05),
}
h36m_labels = {
    0: "Pelvis", 1: "Right hip", 2: "Right knee", 3: "Right ankle", 4: "Left hip", 5: "Left knee", 6: "Left ankle",
    7: "Spine", 8: "Neck", 9: "Nose", 10: "Head", 11: "Left shoulder", 12: "Left elbow", 13: "Left wrist",
    14: "Right shoulder", 15: "Right elbow", 16: "Right wrist",
}
h36m_edges = [
    (0, 1, RIGHT_LEG), (1, 2, RIGHT_LEG), (2, 3, RIGHT_LEG), (0, 4, LEFT_LEG), (4, 5, LEFT_LEG), (5, 6, LEFT_LEG),
    (0, 7, TORSO), (7, 8, TORSO), (8, 9, HEAD), (9, 10, HEAD),
    (8, 11, LEFT_ARM), (11, 12, LEFT_ARM), (12, 13, LEFT_ARM), (8, 14, RIGHT_ARM), (14, 15, RIGHT_ARM), (15, 16, RIGHT_ARM),
]
h36m_offsets = {
    0: (0.00, -0.32), 1: (0.22, -0.15), 2: (0.22, 0.00), 3: (0.22, 0.00), 4: (-0.22, -0.15), 5: (-0.22, 0.00), 6: (-0.22, 0.00),
    7: (0.20, 0.00), 8: (0.26, 0.05), 9: (0.20, 0.00), 10: (0.20, 0.00),
    11: (-0.22, 0.00), 12: (-0.22, 0.00), 13: (-0.22, 0.00), 14: (0.22, 0.00), 15: (0.22, 0.00), 16: (0.22, 0.00),
}

# (c) COMMON15: comparison with MVGFormer
common15_joints = {
    0: (0.00, 4.20), 1: (0.70, 3.90), 2: (0.80, 2.10), 3: (0.75, 0.10), 4: (-0.70, 3.90), 5: (-0.80, 2.10), 6: (-0.75, 0.10),
    7: (0.00, 6.60), 8: (0.00, 8.40),
    9: (-1.20, 6.40), 10: (-1.55, 5.20), 11: (-1.70, 4.05), 12: (1.20, 6.40), 13: (1.55, 5.20), 14: (1.70, 4.05),
}
common15_labels = {
    0: "Pelvis", 1: "Right hip", 2: "Right knee", 3: "Right ankle", 4: "Left hip", 5: "Left knee", 6: "Left ankle",
    7: "Neck", 8: "Nose", 9: "Left shoulder", 10: "Left elbow", 11: "Left wrist",
    12: "Right shoulder", 13: "Right elbow", 14: "Right wrist",
}
common15_edges = [
    (0, 1, RIGHT_LEG), (1, 2, RIGHT_LEG), (2, 3, RIGHT_LEG), (0, 4, LEFT_LEG), (4, 5, LEFT_LEG), (5, 6, LEFT_LEG),
    (0, 7, TORSO), (7, 8, HEAD),
    (7, 9, LEFT_ARM), (9, 10, LEFT_ARM), (10, 11, LEFT_ARM), (7, 12, RIGHT_ARM), (12, 13, RIGHT_ARM), (13, 14, RIGHT_ARM),
]
common15_offsets = {
    0: (0.00, -0.32), 1: (0.22, -0.15), 2: (0.22, 0.00), 3: (0.22, 0.00), 4: (-0.22, -0.15), 5: (-0.22, 0.00), 6: (-0.22, 0.00),
    7: (0.26, 0.05), 8: (0.20, 0.00),
    9: (-0.22, 0.00), 10: (-0.22, 0.00), 11: (-0.22, 0.00), 12: (0.22, 0.00), 13: (0.22, 0.00), 14: (0.22, 0.00),
}

fig, axes = plt.subplots(1, 3, figsize=(17, 8.5))

draw_skeleton_diagram(axes[0], coco_joints, coco_edges, coco_labels, coco_offsets, {0: "center"},
                      "(a) YOLO-Pose / RTMPose (COCO-17)")
draw_skeleton_diagram(axes[1], h36m_joints, h36m_edges, h36m_labels, h36m_offsets, {0: "center", 8: "left"},
                      "(b) TCPFormer / PoseMamba-L (H36M-17)")
draw_skeleton_diagram(axes[2], common15_joints, common15_edges, common15_labels, common15_offsets, {0: "center", 7: "left"},
                      "(c) MVGFormer (COMMON15)")

fig.tight_layout()
save_figure(fig, "fig02_skeleton_conventions")
plt.show()
plt.close(fig)

## 7. Verification Summary and Generated Files

Every table was compared with the values printed in the paper. The tolerances are small: they only allow for the rounding of the paper (two decimals for Table 4 and three for Tables 5 to 7) and, for MVGFormer, for the small numerical differences that can appear when the multi-view inference is run again on a different GPU or software version. A `CHECK` status means that a deviation is larger than that tolerance, and the table above it shows the rows responsible.

In [ ]:
# Verification summary

summary = pd.DataFrame(CHECKS)
display(summary.round(4))

n_ok = int((summary["status"] == "OK").sum())
print(f"\n{n_ok} of {len(summary)} checks passed.")

if n_ok == len(summary):
    print("All tables reproduce the values of the paper.")
else:
    print("Checks to review:", ", ".join(summary.loc[summary["status"] != "OK", "check"]))

summary.to_csv(CSV_DIR / "verification_summary.csv", index=False)

print("\nGenerated files:")
for folder in [TABLES_DIR, FIGURES_DIR, CSV_DIR]:
    files = sorted(path.name for path in folder.iterdir() if path.is_file())
    print(f"\n{folder}  ({len(files)} files)")
    for file in files:
        print("  ", file)

## Troubleshooting

| Message | Cause and solution |
|---|---|
| `No prediction folder found for <model>` | The inference notebook of that model has not been run, or its output is not under `STORAGE_ROOT`. Section 1 prints the folders that are searched. |
| `Ground truth not found for <subject>/<exercise>` or `Calibration not found ...` | The Fit3D ground truth (`Fit3D_2D/<subject>/gt` or `VIDEOS/<subject>/<exercise>/joints3d_25.json`) or the calibration (`camera_params/<subject>`) is missing. |
| `results_per_video_<source>_... not found` (Section 3) | The corresponding run of notebook 01 or 02 with that `POSE_SOURCE` is missing. The table and figures are produced with the available sources. |
| `smoothing_COMMON15_per_video.csv not found` | Run notebook 04 up to its verification section (Section 8). |
| `Filtered prediction not found` | Notebook 04 was run with other filter parameters or the filtered files are missing. The parameters in Section 5 must match `FINAL_FILTER_PARAMS` of notebook 04. |
| Status `CHECK` in the verification summary | The deviation from the paper is larger than the tolerance. Make sure that the monocular predictions come from the overlapping-window inference (`overlap_hann_w243_s60`) and that all 224 sequences of each model were produced with the final notebooks. |